In [1]:
from pcamarillor.spark_utils import SparkUtils
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *

# ✅ Spark 4.0.1 usa Scala 2.13
packages = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0",
    "org.mongodb.spark:mongo-spark-connector_2.13:10.4.0"
])

su = SparkUtils(
    "StreamingApp",
    "spark://spark-master:7077",
    spark_packages=packages
)
spark = su.spark
print("Spark version:", spark.version)

# Leer desde Kafka
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka-1:9092") \
    .option("subscribe", "videogames") \
    .option("startingOffsets", "latest") \
    .load()

schema = StructType([
    StructField("id", StringType()),
    StructField("game", StringType()),
    StructField("genre", StringType()),
    StructField("platform", StringType()),
    StructField("price", DoubleType()),
    StructField("timestamp", TimestampType())
])

df_parsed = df_raw.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*")

# ✅ Prueba primero solo con console antes de escribir a Mongo
query_test = df_parsed.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("numRows", 10) \
    .start()

# query_test.stop()  # Descomentar para detener

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3a645cd9-122c-4659-b16d-86a77cbbe5da;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.0 in central
	found org.apache.kafka#kafka-clients;3.9.0 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central


Spark version: 4.0.1


26/05/05 18:55:40 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e1e2129b-19d5-4fcb-9d21-2d142eec1fb4. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/05 18:55:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/05 18:55:45 WARN ClientUtils: Couldn't resolve server kafka-1:9092 from bootstrap.servers as DNS resolution failed for kafka-1
26/05/05 18:55:45 WARN KafkaOffsetReaderAdmin: Error in attempt 1 getting Kafka offsets: 
org.apache.kafka.common.KafkaException: Failed to create new KafkaAdminClient
	at org.apache.kafka.clients.admin.KafkaAdminClient.createInternal(KafkaAdminClient.java:561)
	at org.apache.kafka.clients.admin.Admin.create(Admin.java:147)
	at org.apac